# v02 · 64-hex address, `(address_hex, length, timestamp)`
`PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster.py` — the shipping baseline. Used by `hyperwebster_layer3.py` (git 2026-04-24), the VAPMIP baseline benchmark (2026-09-25), and every measurement in this paper.

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
hw_mod = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster.py", "hw")
HW = hw_mod.HyperWebster()
print("N =", HW.N)
addr, n, ts = HW.index_text("Hello, World!")
print(addr, n, ts)
assert HW.regenerate_from_address(addr, n) == "Hello, World!"

N = 97
0000000000000000000000000000000000000000002f2a518b0a98eb3978a962 13 Tue Sep 29 23:53:00 2026


## Exact round trips on the corpus sample at several chunk sizes
Address size is compared with the raw 8-bit size.

In [3]:
rows = []
for size in (38, 140, 500, 1000, 2000, 4000):
    t = CORPUS[:size]
    t0 = time.perf_counter(); a = HW.point_to_text(t); t1 = time.perf_counter()
    back = HW.regenerate_text(a); t2 = time.perf_counter()
    assert back == t
    rows.append((size, a.bit_length(), round(a.bit_length()/(8*size), 3), round((t1-t0)*1e3, 2), round((t2-t1)*1e3, 2)))
print("chars  addr_bits  addr/raw  enc_ms  dec_ms")
for r in rows: print("%5d  %9d  %8.3f  %6.2f  %6.2f" % r)
print("log2(97)/8 =", round(math.log2(97)/8, 4))

chars  addr_bits  addr/raw  enc_ms  dec_ms
   38        250     0.822    0.12    0.08
  140        924     0.825    0.65    0.30
  500       3300     0.825    4.19    1.59
 1000       6600     0.825   21.33    5.87
 2000      13199     0.825  123.51   24.46
 4000      26399     0.825  293.62   33.03
log2(97)/8 = 0.825


## The '256-bit' label is a minimum width, not a bound
The hex string is zero-padded to 64 digits; longer strings produce wider hex (the docstring says so).

In [4]:
for size in (10, 38, 39, 40, 200):
    a, n, _ = HW.index_text(CORPUS[:size])
    print(size, "chars ->", len(a), "hex digits")
cap = max(k for k in range(1, 100) if 97**k <= 2**256)
print("largest k with 97^k <= 2^256:", cap, "(97^%d > 2^256)" % (cap+1))

10 chars -> 64 hex digits
38 chars -> 64 hex digits
39 chars -> 65 hex digits
40 chars -> 66 hex digits
200 chars -> 330 hex digits
largest k with 97^k <= 2^256: 38 (97^39 > 2^256)


## Two loops, one address, very different cost
`point_to_text` computes `N ** i` for every position i (`address += index * N**i`). A forward Horner loop (`address = address * N + index`, the form used in v05's `HyperIndex`, v09 and the ValaQuenta module) returns the identical integer with one small multiplication per character.

In [5]:
import random
rnd = random.Random(1)
def horner_forward(t, N=HW.N, idx=HW._char_index):
    a = 0
    for c in t: a = a * N + idx[c] + 1
    return a - 1
rows = []
for n in (1000, 2000, 4000, 8000, 16000):
    t = "".join(rnd.choice(HW.characters) for _ in range(n))
    a0 = time.perf_counter(); x = HW.point_to_text(t); a1 = time.perf_counter(); y = horner_forward(t); a2 = time.perf_counter()
    assert x == y
    rows.append((n, a1 - a0, a2 - a1))
print("chars   point_to_text s   forward Horner s   ratio")
for n, p, f in rows: print("%5d   %15.4f   %16.4f   %5.1f" % (n, p, f, p / f))
e = lambda k: math.log(rows[-1][k] / rows[0][k]) / math.log(rows[-1][0] / rows[0][0])
print("fitted exponent: point_to_text %.2f | forward Horner %.2f" % (e(1), e(2)))

chars   point_to_text s   forward Horner s   ratio
 1000            0.0091             0.0007    13.3
 2000            0.0464             0.0026    18.1
 4000            0.2924             0.0099    29.7
 8000            1.6354             0.0319    51.3
16000           10.1590             0.1122    90.6
fitted exponent: point_to_text 2.53 | forward Horner 1.84


Between 1,000 and 16,000 characters the time grows as about n^2.5 for `point_to_text` (2.4–2.6 across runs) and n^1.8 for the forward loop, and the ratio between them widens with size. The 4.7–5.3× growth per doubling recorded by the VAPMIP baseline benchmark, which runs this class, is the first exponent.

## Unrelated but on the same file
The module smoke-test reads `PTOL_ROOT`, which is undefined; the class itself is unaffected.

In [6]:
import inspect
print("PTOL_ROOT referenced in __main__ block:", "PTOL_ROOT" in inspect.getsource(hw_mod))
print("defined in module namespace:", hasattr(hw_mod, "PTOL_ROOT"))

PTOL_ROOT referenced in __main__ block: True
defined in module namespace: False
